# CS 195: Natural Language Processing
## PyTorch Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_2_PyTorchEmbeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [1]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers

## Reorganization of where we left off from last time

In [2]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random 
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


def prepare_one_hot_inputs(couples, labels, vocabulary_size):
    inputs = []
    for target_word, context_word in couples:
        target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
        context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
        inputs.append(torch.cat([target_one_hot, context_one_hot]))

    inputs_array = torch.stack(inputs).float()
    labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return inputs_array, labels_array


def train_embedding_model(inputs_array, labels_array, vocabulary_size):

    embedding_model = nn.Sequential(
        nn.Linear(vocabulary_size * 2, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

    for epoch in range(20000):
        logits = embedding_model(inputs_array)
        loss = loss_fn(logits, labels_array)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo



In [ ]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for example in sentences:
    ids = tokenizer.encode(example).ids
    tokens = tokenizer.encode(example).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())





Here's an example of some tokens: ['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 1000, loss=0.3978
epoch 2000, loss=0.2550
epoch 3000, loss=0.1886
epoch 4000, loss=0.1408
epoch 5000, loss=0.1039
epoch 6000, loss=0.0780
epoch 7000, loss=0.0612
epoch 8000, loss=0.0512
epoch 9000, loss=0.0453
epoch 10000, loss=0.0418
epoch 11000, loss=0.0397
epoch 12000, loss=0.0385
epoch 13000, loss=0.0378
epoch 14000, loss=0.0374
epoch 15000, loss=0.0372
epoch 16000, loss=0.0370
epoch 17000, loss=0.0369
epoch 18000, loss=0.0369
epoch 19000, loss=0.0369
epoch 20000, loss=0.0368


In [4]:

cats_embedding = get_embedding("cats", tokenizer, embedding_model)
dogs_embedding = get_embedding("dogs", tokenizer, embedding_model)
ocean_embedding = get_embedding("ocean", tokenizer, embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([ 0.8572, -0.8290,  0.9891,  0.6411,  0.8919, -0.8285,  0.8449,  0.9637,
         0.9841,  0.7796,  0.3202,  0.9332,  0.9624, -0.9470,  0.4674,  1.0327,
         0.6354,  0.9318,  0.8774,  1.0425,  0.9880,  0.6924,  0.9949, -0.9327,
         0.9513, -0.9992, -0.5045,  0.8357,  0.9678,  1.0250,  0.9761,  0.5013,
         0.5515,  0.9317, -0.0515,  0.8429,  0.7618,  0.9055,  0.9622,  0.9060,
         0.1177, -0.9510,  0.9147,  0.8748,  0.9655,  0.8377, -1.2041,  0.9136,
        -0.8472,  0.9244])
tensor([-0.3154,  1.0533,  0.9579, -0.4324,  0.1969,  0.9609,  0.9421, -0.7270,
        -0.7239,  0.8792,  0.9850,  0.8479,  0.9509,  1.0006,  0.9216,  0.9764,
        -0.7204, -0.5771,  0.9324, -0.9319, -0.6583,  0.7122,  0.9904,  0.6740,
        -1.0051,  1.0195,  1.0529, -0.9895, -0.9905,  0.9834, -0.3299,  0.7908,
         0.6512, -0.9712,  0.5721,  0.3194, -0.6363, -0.0161,  0.9605, -0.6840,
        -0.7028, -0.4447,  0.9591,  0.9238, -0.6357,  0.9103, -0.0707,  0.7355,
         0.98

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [5]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])

/home/evan/evan-drake/cs-195/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Group Exercise

Try creating word embeddings for the AG News dataset. *Note that you will only be able to do a very small subset with this code. Start with 10 examples and work your way up*

* How big of a vocabulary did you need to use to get reasonable tokens for this data?
* Once you use more examples, you'll start to notice that you're going to run out of memory and the kernel will crash. What do you think the issue is? Can you think of a way to do the same idea but keep less data in memory at any given time?



In [15]:
examples = data["train"]["text"][0:15]

tokenizer = train_tokenizer(examples, vocabulary_size=1000)

tokenized_sentences = []
for example in examples:
    ids = tokenizer.encode(example).ids
    tokens = tokenizer.encode(example).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())




Here's an example of some tokens: ['Dollar', 'Falls', 'Broadly', 'on', 'Record', 'Trade', 'Gap', 'NEW', 'YORK', '(', 'Reuters', ')', '-', 'The', 'dollar', 'tumbled', 'broadly', 'on', 'Friday', 'after', 'data', 'showing', 'a', 'record', 'U', '.', 'S', '.', 'trade', 'deficit', 'in', 'June', 'cast', 'fresh', 'doubts', 'on', 'the', 'economy', "'", 's', 'recovery', 'and', 'its', 'ability', 'to', 'draw', 'foreign', 'capital', 'to', 'fund', 'the', 'growing', 'gap', '.']
epoch 1000, loss=0.4429
epoch 2000, loss=0.3173
epoch 3000, loss=0.2519
epoch 4000, loss=0.1941
epoch 5000, loss=0.1414
epoch 6000, loss=0.0987
epoch 7000, loss=0.0679
epoch 8000, loss=0.0476
epoch 9000, loss=0.0349
epoch 10000, loss=0.0271
epoch 11000, loss=0.0225
epoch 12000, loss=0.0197
epoch 13000, loss=0.0180
epoch 14000, loss=0.0170
epoch 15000, loss=0.0164
epoch 16000, loss=0.0160
epoch 17000, loss=0.0158
epoch 18000, loss=0.0156
epoch 19000, loss=0.0156
epoch 20000, loss=0.0155


## The PyTorch Embedding Layer
PyTorch provides an `nn.Embedding` layer, which is a linear layer with a lookup table. It allows you to input a token id and look up a row vector (akin to the linear node associated with that id).

It's mathematically equivalent to the one-hot encoding + `nn.Linear` layer we saw before, but it much more memory efficient - we don't have to waste space storing all of those 0s. 

Let's try the same experiment as before but using the `nn.Embedding` layer. 


In [20]:
def train_embedding_model_v2(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)
    skipgram_classifier = nn.Linear(2*50, 1) # target emb + context emb

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(list(embedding_model.parameters()) + list(skipgram_classifier.parameters()), lr=0.0003) # concatenate the parameters for the embedding model and skipgram classifier

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(5000):
        target_ids = pair_ids[:, 0] # [N]
        context_ids = pair_ids[:, 1] # [N]

        target_embeddings = embedding_model(target_ids)  # [N, 50]
        context_embeddings = embedding_model(context_ids)  # [N, 50]

        target_context_together = torch.cat([target_embeddings, context_embeddings], dim=1)  # [N, 100]

        logits = skipgram_classifier(target_context_together)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 500 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding_v2(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for example in sentences:
    ids = tokenizer.encode(example).ids
    tokens = tokenizer.encode(example).tokens
    tokenized_sentences.append(ids)

print(tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)
embedding_model_v2 = train_embedding_model_v2(couples, labels, vocabulary_size=tokenizer.get_vocab_size())




['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 500, loss=0.4202
epoch 1000, loss=0.2900
epoch 1500, loss=0.2623
epoch 2000, loss=0.2536
epoch 2500, loss=0.2501
epoch 3000, loss=0.2485
epoch 3500, loss=0.2476
epoch 4000, loss=0.2470
epoch 4500, loss=0.2467
epoch 5000, loss=0.2465


In [21]:

cats_embedding = get_embedding_v2("cats", tokenizer, embedding_model_v2)
dogs_embedding = get_embedding_v2("dogs", tokenizer, embedding_model_v2)
ocean_embedding = get_embedding_v2("ocean", tokenizer, embedding_model_v2)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)

tensor([-2.2201e+00,  7.6761e-01,  1.7594e-01, -6.7373e-01,  6.5258e-02,
         3.5715e-01,  3.9052e-01,  7.0534e-01,  1.3884e+00,  1.6264e+00,
        -1.0263e+00, -2.4427e-04, -1.8798e+00,  8.8259e-01, -1.9989e+00,
         8.9765e-02, -5.4493e-01,  1.0953e+00,  2.0069e-01,  2.4923e+00,
        -1.0659e+00, -8.4782e-01, -2.7224e+00, -5.9250e-01,  5.8377e-01,
        -8.7593e-01, -2.3170e-01,  3.3350e+00, -7.7511e-01,  5.1351e-01,
        -2.6407e-01,  8.4358e-01,  1.4770e-02,  8.4625e-01,  4.9336e-01,
        -7.3821e-01, -7.7347e-01, -5.5934e-01,  1.1048e+00, -9.8999e-01,
         1.3116e-01,  2.2476e+00, -7.4817e-01, -1.7025e-02, -1.8024e-01,
        -1.3828e+00, -8.7220e-01,  1.1373e+00,  4.5576e-01,  9.2639e-01])
tensor([-1.1808, -1.1962,  0.2515,  0.9783, -0.7215,  0.1855,  1.6578,  1.4440,
         1.2561, -0.0834, -0.9566,  0.3688, -1.2052, -1.1098,  0.5544,  0.6383,
         0.5355,  0.4215, -1.1126,  0.1302,  1.3901, -1.3019, -0.3241, -2.7800,
        -0.3036,  0.1851, -1.

### Caveat

With such a small dataset, we're not actually getting good embeddings yet. 

## Applied Exploration

Create word embeddings for a larger portion of the AG News dataset, say 5000 texts

Show some example word embeddings for some words that appear in the dataset (*cats* and *dogs* may not be good examples for this one)

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* What learning rate and number of training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

## Adding an Embedding layer to your model for other learning tasks

First, let's prepare the data
* We could use the same kind of tokenizer, or just use an existing one - let's go back to doing it with a pretrained Hugging Face tokenizer


In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

data = load_dataset("ag_news")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # for some reason this tokenizer doesn't have a pad token by default, but we need one to be able to batch our inputs together, so we'll just add a new special token for padding

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)


## Model with an Embedding layer

We'll set up the embedding layer and sequential model as before

The optimizer needs to have a concatenation of the parameters for both parts

In [9]:
embedding = nn.Embedding(len(hf_tokenizer), 50) # len(hf_tokenizer) is the vocab size including the new padding token
classifier = nn.Sequential(
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.01)

## Training Loop with Embedding layer

The input goes into the embedding layer
* returns an embedding for each words, so we aggregate them with the *mean* to get an embedding for the whole training example

In [23]:


for epoch in range(100):
    optimizer.zero_grad()

    emb = embedding(X_train) # [batch_size, seq_len, embedding_dim]
    pooled_emb = emb.mean(dim=1) # simple way to get a single vector [batch_size, embedding_dim] for the whole sequence - just average the token embeddings
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")



Epoch 1, loss = 1.3808
Epoch 11, loss = 1.2952
Epoch 21, loss = 0.9118
Epoch 31, loss = 0.5034
Epoch 41, loss = 0.3348
Epoch 51, loss = 0.2470
Epoch 61, loss = 0.1914
Epoch 71, loss = 0.1539
Epoch 81, loss = 0.1262
Epoch 91, loss = 0.1044


## Evaluating works similarly

In [24]:
with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.9053947329521179


## Applied Exploration

Perform a text classification experiment with another classification data set
* Try different embedding vector lengths (other than just 50 as we did here)
* Experiment with different neural network structures, learning rates, and number of epochs

Report your results and reflect on them
* Describe your dataset
* Describe what you did
* Report the results you observed
* Discuss any interesting insights